##### structured output

###### Models can be requsted to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

##### pydantic


###### pydantic models provide the richest feature set with field validation ,descriptions , and nested structures.

In [1]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")




In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash")

model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x7548ff9bfe00>, default_metadata=(), model_kwargs={})

In [5]:
from pydantic import BaseModel , Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")
    

In [7]:
model_with_structure=model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x79412aa8b250>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'d

In [9]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='*Inception* is a critically acclaimed and commercially successful 2010 science fiction action film written, directed, and produced by Christopher Nolan. It\'s renowned for its complex narrative, mind-bending visuals, and exploration of dreams, reality, and the human subconscious.\n\nHere\'s a detailed breakdown:\n\n---\n\n### **1. Basic Premise**\n\nThe film centers on **Dominick "Dom" Cobb** (Leonardo DiCaprio), a skilled "extractor" who steals valuable information by entering people\'s dreams. His unique ability has made him a fugitive, unable to return home to his children. He\'s offered a chance at redemption: instead of extracting an idea, he must perform "inception" – planting an idea into a target\'s subconscious mind. If successful, his criminal record will be cleared, allowing him to reunite with his family.\n\n---\n\n### **2. The Core Mission: Inception**\n\n*   **The Client:** Cobb is hired by **Saito** (Ken Watanabe), a powerful businessman, to perform in

In [14]:
responce=model_with_structure.invoke("Provide details about the movie Inception")

responce

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [17]:
responce.year

2010

#### Message output alongside parsed structure 

In [20]:
from pydantic import BaseModel , Field

class Movie(BaseModel):
    """A movie with details ."""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

responce = model_with_structure.invoke("Provide details about the movie Inception")

responce

    

{'raw': AIMessage(content='{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f464d-e368-7902-91c6-963a10be5838-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 109, 'total_tokens': 117, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 78}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

#### Nested Structure

In [21]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]    
    budget:float | None = Field(None,description="Budged in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)


In [22]:
model_with_structure.invoke("Provide details about the movie Inception")

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

#### TypedDict

###### TypedDict provides a simpler alternative using python's build-in typing , ideal when we don't need runtime validation .

In [3]:
from typing_extensions import  TypedDict , Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


  


In [4]:
model_with_typedDist_structure = model.with_structured_output(MovieDict)

In [6]:
model_with_typedDist_structure.invoke("peovide the details of the movie Avengers")

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [7]:
model.profile

{'name': 'Gemini 2.5 Flash',
 'release_date': '2025-03-20',
 'last_updated': '2025-06-05',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

##### DataClasses

###### A data class is a class typicall containing mainly data  although there aren't relly any restrictions. you create it using the @dataclass decorator

In [9]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [ ]:
from langchain.agents import create_agent
from pydantic import BaseModel , Field

class ContactInfo(BaseModel):
    """contact information for a person"""
    name:str = Field(description="the name of the person")
    email:str = Field(description="the email of the person")
    phone:str = Field(description="the phone number of the person")

agent = create_agent(
    model="google_genai:gemini-3.5-flash" ,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{
        "role":"user" ,
        "content":"Extract contact info from : John Doe ,john@example.com,9897765432"
    }]
})



ContactInfo(name='John Doe', email='john@example.com', phone='9897765432')

In [4]:
result

{'messages': [HumanMessage(content='Extract contact info from : John Doe ,john@example.com,9897765432', additional_kwargs={}, response_metadata={}, id='0f997b30-1d4c-497c-bc4b-fa861388d8e5'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"john@example.com","phone":"9897765432"}', 'extras': {'signature': 'EsALCr0LARFNMg/LgWiZdx8H+HYGPmYNHzD2gtmxgDN1RoyZZIeeGnHbVt65MlDZXMNKWwxOr9yLg+2qybp3E00plygxkaJSRJcSylwi9gUoyTCb+Hp79aGifcO0YSakr9OdLtS+p9t5CpBoyl5o0gLIKBz3JXfVLC346WRpFWLHVf61btYGFEjrdXAA58dG3jAcgmSNDNtHVPbUqyEAkF8xKjkYKM+ID42AOAcTOP4iad0TpmjORsnbDPcz2C/Dcm69Z1vfV8PJ6PGj1t2d/5jagR+Vnk1qaB/P/8WbTOhZQXxYPncRNJ/Ddi/f+HZdmx6P/EYCmXP/TVVaylNkwKCVsH4y6NTXaGodkf/Sr63B2ftPY57Mwsy3FkTudsEi0yTocv84wwLxFpP4kP4BXOXck5cjLU7URY3KTBCCV/zHJtOzdscuINUBO+ZBM7JnMyfMztSOApXzADhfG+XIqUoQBORsontjLG8R8d5Hfjh9ORKuVC756LRoozi4Z4roMZBi52XMCAp3ajQCBRBwfq18hsuHaQjFP4XKQXcsF3wi5rYfmeXTfDda28Uq7prkxeUJAFH50kPuLB75Dc+iL4kT2bI3k2PArNcktFUG3dZ8E+rAhG+CiwSPDeGMn7w8UoENRoPBIkhx7v1BKjD0XZYg5V

In [5]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='9897765432')

#### Data class

In [5]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """contact info for a person"""
    name:str #The name of the person
    email:str #The email address of the person
    phone:str #the phone number of the perosn 

agent = create_agent(
    model="google_genai:gemini-3.5-flash" ,
    response_format=ContactInfo
)

In [7]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Harsh is a 23-year-old boy with email harsh@gmail.com and mobile number 9896982345."
        }
    ]
})

print(result)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}